# 1. Import and config

In [1]:
import os, glob, math, random
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import orjson as json 
except Exception:
    import json


MPD_DIR        = "/kaggle/input/spotify-million/data"
SPOTIFY12M_CSV = "/kaggle/input/spotify-12m-songs/tracks_features.csv"
OUT_PATH       = "/kaggle/working/interactions_mpd.csv"


SAMPLE_FRAC          = 1.0     # 1.0 = all MPD slices
MIN_PLAYLIST_LEN     = 5
NEGATIVES_PER_POS    = 1
MAX_NEG_PER_PLAYLIST = 5
RANDOM_SEED          = 42

SESSION_STEP_SEC     = 2 * 60 * 60   # 2h per edit session
INTRA_STEP_SEC       = 120           # 2min between tracks within a session

# num_edits fallback (when missing)
AVG_TRACKS_PER_SESSION = 10
MAX_SESSIONS           = 32

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 1.1 Some helper functions

In [2]:
def extract_track_id(uri: str):
    # "spotify:track:<id>" -> "<id>"
    if isinstance(uri, str) and isinstance(uri, (str,)):
        if uri.startswith("spotify:track:"):
            return uri.split(":")[2]
    return None

def estimate_num_edits(n_tracks: int) -> int:
    if not isinstance(n_tracks, int) or n_tracks <= 0:
        return 1
    est = math.ceil(n_tracks / AVG_TRACKS_PER_SESSION)
    return int(max(1, min(MAX_SESSIONS, est)))

# 2. Load dataset

## 2.1 Spotify 1.2 million

In [3]:
print("Loading Spotify-12M IDs …")
items_ids = pd.read_csv(SPOTIFY12M_CSV, usecols=["id"], dtype={"id": "string"})
items_ids = items_ids.dropna().drop_duplicates()
items_ids["id"] = items_ids["id"].str.strip()
valid_ids_set = set(items_ids["id"].astype(str).values)
valid_ids_arr = np.array(list(valid_ids_set))
N_ITEMS = len(valid_ids_arr)
print(f"Spotify-12M unique item_ids: {N_ITEMS:,}")

Loading Spotify-12M IDs …
Spotify-12M unique item_ids: 1,204,025


## 2.2 Million Playist + Filtering

I load million playist, filtering it (min allowed playlist len, check whether id exists in track dataset or not) and estimate timestamp (based on num of edit and session).

In [ ]:
print("Parsing MPD slices + building positives")
files = sorted(glob.glob(os.path.join(MPD_DIR, "mpd.slice.*.json")))
if not files:
    raise SystemExit(f"No MPD slices found in: {MPD_DIR}")

if SAMPLE_FRAC < 1.0:
    k = max(1, int(len(files) * SAMPLE_FRAC))
    files = files[:k]  # deterministic head sample
    print(f"Using {k} / {len(glob.glob(os.path.join(MPD_DIR, 'mpd.slice.*.json')))} files")

pos_rows = []  # Accumulating tuples for one big DataFrame build at the end

for path in tqdm(files, desc="Reading MPD files", unit="file"):
    with open(path, "rb") as f:
        data = json.loads(f.read())
    for pl in data.get("playlists", []):
        pid = pl.get("pid")
        if pid is None:
            continue

        tracks = pl.get("tracks", []) or []
        if MIN_PLAYLIST_LEN and len(tracks) < MIN_PLAYLIST_LEN:
            continue

        user_id     = f"mpd_pid_{pid}"
        modified_at = int(pl.get("modified_at", 0))

        ne = pl.get("num_edits")
        if isinstance(ne, int) and ne > 0:
            num_edits = ne
        else:
            # estimate from playlist length
            n_tracks_field = pl.get("num_tracks")
            playlist_len = int(n_tracks_field) if isinstance(n_tracks_field, int) else len(tracks)
            num_edits = estimate_num_edits(playlist_len)
        num_edits = max(1, min(MAX_SESSIONS, int(num_edits)))

        L = len(tracks)
        chunk = max(math.ceil(L / num_edits), 1)

        for tr in tracks:
            tid = extract_track_id(tr.get("track_uri"))
            if not tid or tid not in valid_ids_set:
                continue
            pos = int(tr.get("pos", 0))
            sess_idx  = min(pos // chunk, num_edits - 1)
            within_ix = pos % chunk
            event_ts  = int(modified_at - sess_idx * SESSION_STEP_SEC - within_ix * INTRA_STEP_SEC)
            pos_rows.append((user_id, tid, 1, event_ts))

pos_df = pd.DataFrame(pos_rows, columns=["user_id","item_id","exists","timestamp"])
print(f"Positives: {len(pos_df):,} | users={pos_df['user_id'].nunique():,} | items={pos_df['item_id'].nunique():,}")

Parsing MPD slices + building positives


Reading MPD files:  13%|█▎        | 129/1000 [00:43<04:35,  3.16file/s]

# 3. Negative sampling

Playlist let me know what song exist (positive/like), but no information about what song that person not choose. So, to make an dataset similar to an interaction dataset, I use negative sampling (negative/dislike).

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(valid_ids_arr)

def start_offset(uid: str) -> int:
    return (hash(uid) & 0xFFFFFFFF) % N_ITEMS

neg_rows = []
for uid, g in tqdm(pos_df.groupby("user_id", sort=False),
                   total=pos_df["user_id"].nunique(),
                   desc="Sampling negatives", unit="playlist"):
    pos_set = set(g["item_id"])
    if not pos_set:
        continue

    target_k = int(len(g) * NEGATIVES_PER_POS)
    target_k = min(target_k, MAX_NEG_PER_PLAYLIST, N_ITEMS - len(pos_set))
    if target_k <= 0:
        continue

    ts_med = int(g["timestamp"].median())
    taken = 0
    picked = []
    idx = start_offset(uid)

    while taken < target_k:
        cand = valid_ids_arr[idx]
        if cand not in pos_set:
            picked.append(cand)
            taken += 1
        idx += 1
        if idx == N_ITEMS:
            idx = 0

    neg_rows.extend((uid, nid, 0, ts_med) for nid in picked)

neg_df = pd.DataFrame(neg_rows, columns=["user_id","item_id","exists","timestamp"])
print(f"Negatives: {len(neg_df):,}")

# 4. Save dataset and check

In [ ]:
final_df = pd.concat([pos_df, neg_df], ignore_index=True)
final_df = final_df.astype({"user_id": "string", "item_id": "string", "exists": int, "timestamp": int})

print(f"Final interactions: {len(final_df):,} | users={final_df['user_id'].nunique():,} | items={final_df['item_id'].nunique():,}")
final_df.to_csv(OUT_PATH, index=False)
print("Saved ->", OUT_PATH)

In [ ]:
df = pd.read_csv("/kaggle/working/interactions_mpd.csv")

In [ ]:
df[df['exists'] == 0]